# TP1 – Continual Learning on Seq-CIFAR-10
**I309 – Visión Artificial Avanzada · Universidad de San Andrés**

This notebook is the single entry point for all experiments.
It follows the four stages of the assignment:

1. Dataset preparation (Seq-CIFAR-10, replay buffer)
2. Pre-training with Supervised Contrastive Learning (SupCon)
3. Continual Learning methods: Naive, EWC, LwF, Co²L
4. Comparison of results (Class-IL and Task-IL)


## 0. Setup

In [1]:
import sys, os

# __file__ doesn't exist in Jupyter — use the working directory instead.
# Launch Jupyter from the repo root (the folder containing tp1.ipynb) and
# this will resolve correctly.
REPO_ROOT = os.path.abspath('')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'REPO_ROOT: {REPO_ROOT}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

PyTorch  : 2.2.2+cpu
REPO_ROOT: c:\Users\Hp\OneDrive\Documentos\UdeSA\5_ano\Vision_artificial_avanzada\TP1\Continual-Learning-on-Seq-CIFAR-10
Device   : cpu


In [2]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from data.dataset import SeqCIFAR10, TASK_CLASSES, CLASS_NAMES
from data.buffer import ReplayBuffer
from models.backbone import Backbone
from losses.supcon import SupConLoss
from losses.distillation import AsymmetricDistillationLoss
from methods.naive import NaiveFineTuning
from methods.ewc import EWC
from methods.lwf import LwF
from methods.co2l import Co2L
from methods.trainer import ContinualTrainer
from utils.metrics import evaluate_class_il, evaluate_task_il, compute_forgetting, MetricsTracker
from utils.visualization import plot_accuracy_curve, plot_forgetting_curve, plot_embeddings, plot_loss_curve

print('All imports OK')

All imports OK


## Stage 1 – Dataset Preparation

Split CIFAR-10 into 5 sequential tasks (2 classes each) and set up the replay buffer.

| Task | Classes               | Train images |
|------|-----------------------|--------------|
| 0    | airplane, automobile  | 10 000       |
| 1    | bird, cat             | 10 000       |
| 2    | deer, dog             | 10 000       |
| 3    | frog, horse           | 10 000       |
| 4    | ship, truck           | 10 000       |

In [3]:
DATA_ROOT   = './data'   # torchvision downloads CIFAR-10 here on first run
N_TASKS     = 5
BATCH_SIZE  = 128
BUFFER_SIZE = 200        # replay buffer capacity across all past tasks

seq_cifar = SeqCIFAR10(data_root=DATA_ROOT, n_tasks=N_TASKS, batch_size=BATCH_SIZE)
buffer    = ReplayBuffer(capacity=BUFFER_SIZE)

print(f'Task splits (class indices): {TASK_CLASSES}')
print()
for t in range(N_TASKS):
    names = seq_cifar.get_class_names(t)
    n_train = seq_cifar.task_size(t, train=True)
    n_test  = seq_cifar.task_size(t, train=False)
    print(f'  Task {t}: {names[0]:12s} + {names[1]:12s}  |  train={n_train:5d}  test={n_test:4d}')

100%|██████████| 170498071/170498071 [00:37<00:00, 4535451.23it/s]


Extracting ./data\cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Task splits (class indices): [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

  Task 0: airplane     + automobile    |  train=10000  test=2000
  Task 1: bird         + cat           |  train=10000  test=2000
  Task 2: deer         + dog           |  train=10000  test=2000
  Task 3: frog         + horse         |  train=10000  test=2000
  Task 4: ship         + truck         |  train=10000  test=2000


In [ ]:
# --- Verify class filtering is correct ---
print('Verifying class filtering...')
for t in range(N_TASKS):
    c0, c1 = seq_cifar.get_classes(t)
    loader = seq_cifar.get_task_loader(t, train=True)
    seen_classes = set()
    for _, labels in loader:
        seen_classes.update(labels.tolist())
    assert seen_classes == {c0, c1}, f'Task {t}: expected {{{c0},{c1}}}, got {seen_classes}'
    print(f'  Task {t} OK — classes {seen_classes} = {seq_cifar.get_class_names(t)}')

print()
# Verify a SupCon loader returns (B, 2, C, H, W)
supcon_loader = seq_cifar.get_task_loader(0, train=True, supcon=True)
imgs, labels = next(iter(supcon_loader))
assert imgs.ndim == 5 and imgs.shape[1] == 2 and imgs.shape[2:] == (3, 32, 32), \
    f'Expected (B,2,3,32,32) but got {imgs.shape}'
print(f'SupCon loader batch shape: {imgs.shape}  →  (B={imgs.shape[0]}, 2 views, C=3, H=32, W=32)')
print()
print('All checks passed.')

## Stage 2 – Pre-training with Supervised Contrastive Learning

Train the backbone encoder on Task 0 using SupCon loss, then attach and train a linear classification head.

In [ ]:
SUPCON_EPOCHS    = 200
SUPCON_LR        = 0.5
LINEAR_EPOCHS    = 100

backbone = Backbone(num_classes=10).to(DEVICE)
print(backbone)

# --- Quick shape check ---
dummy = torch.zeros(4, 3, 32, 32, device=DEVICE)
assert backbone(dummy).shape              == (4, 10),        "forward() shape"
assert backbone.encode(dummy).shape       == (4, 512),       "encode() shape"
assert backbone.project(dummy).shape      == (4, 128),       "project() shape"

dummy_2v = torch.zeros(4, 2, 3, 32, 32, device=DEVICE)
assert backbone.forward_supcon(dummy_2v).shape == (4, 2, 128), "forward_supcon() shape"

print('Shape checks OK')
print(f'Encoder frozen: {backbone.is_encoder_frozen}')

In [ ]:
import copy

def make_eval_fn(seq_cifar, device, tracker: MetricsTracker):
    """Return an eval_fn for ContinualTrainer that feeds MetricsTracker.

    After each task t:
      - Builds test loaders for tasks 0..t.
      - Runs Class-IL (no task hint, argmax over 10 classes).
      - Runs Task-IL (task hint: mask to 2 task classes, then argmax).
      - Records full per-task accuracies in the tracker.
      - Returns avg metrics for ContinualTrainer's console log.
    """
    def eval_fn(task_id: int, method):
        test_loaders = [seq_cifar.get_task_loader(t, train=False)
                        for t in range(task_id + 1)]
        class_il = evaluate_class_il(method.backbone, test_loaders, device)
        task_il  = evaluate_task_il(method.backbone,  test_loaders, device)
        tracker.update(task_id, class_il, task_il)
        return {"class_il": class_il["avg_acc"], "task_il": task_il["avg_acc"]}
    return eval_fn

In [ ]:
# --- 3.1 Naive Fine-Tuning ------------------------------------------------
naive_tracker  = MetricsTracker(n_tasks=N_TASKS)
naive_backbone = copy.deepcopy(backbone)
naive          = NaiveFineTuning(naive_backbone, device=DEVICE, lr=0.01)
naive_trainer  = ContinualTrainer(
    naive, seq_cifar, n_epochs=50,
    eval_fn=make_eval_fn(seq_cifar, DEVICE, naive_tracker),
)
naive_results = naive_trainer.train_all_tasks()
print(naive_tracker)
naive_tracker.summary_df()

In [ ]:
# --- 3.2 EWC --------------------------------------------------------------
# Protocol: no replay — penalty term on Fisher matrix prevents forgetting.
# TODO: uncomment once EWC.train_task / end_task are implemented.

# ewc_backbone = copy.deepcopy(backbone)
# ewc = EWC(ewc_backbone, device=DEVICE, ewc_lambda=400.0)
# ewc_trainer = ContinualTrainer(ewc, seq_cifar, n_epochs=50, eval_fn=eval_fn)
# ewc_results = ewc_trainer.train_all_tasks()
# ewc_class_il = [ewc_results["eval_results"][t]["class_il"] for t in range(N_TASKS)]
# ewc_task_il  = [ewc_results["eval_results"][t]["task_il"]  for t in range(N_TASKS)]

In [ ]:
# --- 3.3 LwF --------------------------------------------------------------
# Protocol: no replay — knowledge distillation from the previous model.
# TODO: uncomment once LwF.train_task is implemented.

# lwf_backbone = copy.deepcopy(backbone)
# lwf = LwF(lwf_backbone, device=DEVICE)
# lwf_trainer = ContinualTrainer(lwf, seq_cifar, n_epochs=50, eval_fn=eval_fn)
# lwf_results = lwf_trainer.train_all_tasks()
# lwf_class_il = [lwf_results["eval_results"][t]["class_il"] for t in range(N_TASKS)]
# lwf_task_il  = [lwf_results["eval_results"][t]["task_il"]  for t in range(N_TASKS)]

In [ ]:
# --- 3.4 Co²L -------------------------------------------------------------
# Protocol: replay-enabled — SupCon on current-task data + buffer samples.
# uses_replay=True is declared on Co2L; ContinualTrainer logs this per task.
# TODO: uncomment once Co2L.train_task / end_task are implemented.

# co2l_backbone = copy.deepcopy(backbone)
# co2l_buffer   = ReplayBuffer(capacity=BUFFER_SIZE)
# co2l = Co2L(co2l_backbone, device=DEVICE, replay_buffer=co2l_buffer)
# co2l_trainer = ContinualTrainer(co2l, seq_cifar, n_epochs=500, eval_fn=eval_fn)
# co2l_results = co2l_trainer.train_all_tasks()
# co2l_class_il = [co2l_results["eval_results"][t]["class_il"] for t in range(N_TASKS)]
# co2l_task_il  = [co2l_results["eval_results"][t]["task_il"]  for t in range(N_TASKS)]

## Stage 4 – Comparison of Results

In [ ]:
# --- Stage 4: Comparison of Results --------------------------------------
# Collect avg-accuracy curves from each tracker (uncomment as methods are done)
all_trackers = {
    "Naive": naive_tracker,
    # "EWC":   ewc_tracker,
    # "LwF":   lwf_tracker,
    # "Co2L":  co2l_tracker,
}

# Accuracy curves: avg Class-IL / Task-IL vs. number of tasks learned
plot_accuracy_curve(
    {m: t.avg_acc_curve("class_il") for m, t in all_trackers.items()},
    scenario="Class-IL",
)
plot_accuracy_curve(
    {m: t.avg_acc_curve("task_il") for m, t in all_trackers.items()},
    scenario="Task-IL",
)

# Forgetting curves
plot_forgetting_curve(
    {m: t.forgetting("class_il") for m, t in all_trackers.items()},
)

In [ ]:
# Final summary table: one row per method, final Class-IL / Task-IL / forgetting
import pandas as pd

rows = []
for method_name, tracker in all_trackers.items():
    rows.append({
        "Method":         method_name,
        "Class-IL":       round(tracker.avg_acc_curve("class_il")[-1], 4),
        "Task-IL":        round(tracker.avg_acc_curve("task_il")[-1],  4),
        "Avg Forgetting": round(tracker.avg_forgetting("class_il"),    4),
    })

pd.DataFrame(rows).set_index("Method")